In [ ]:
# --- Arranque del entorno local (en Google Colab no cambia nada) ---
import pathlib
import sys
import types

try:
    _raiz = next(
        d
        for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
        if (d / "curso_setup.py").exists()
    )
    sys.path.insert(0, str(_raiz))
    import curso_setup
except StopIteration:  # Google Colab: se usa un sustituto mínimo
    import subprocess

    def _clonar(destino="curso_IA_CHEC"):
        if not pathlib.Path(destino).is_dir():
            subprocess.run(
                ["git", "clone", "https://github.com/UN-GCPDS/curso_IA_CHEC.git", destino],
                check=True,
            )
        return pathlib.Path(destino)

    def _descargar(file_id, destino):
        if not pathlib.Path(destino).exists():
            import gdown

            gdown.download(id=file_id, output=destino, quiet=False)
        return pathlib.Path(destino)

    curso_setup = types.SimpleNamespace(
        en_colab=lambda: True,
        init=lambda *a, **k: pathlib.Path.cwd(),
        clonar_curso=_clonar,
        descargar_drive=_descargar,
    )

curso_setup.init()

In [ ]:
if curso_setup.en_colab():
    !pip install -U streamlit langchain langchain-openai langchain-experimental

In [ ]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
import os

In [ ]:
eventos_df=pd.read_pickle("Eventos_transformador.pkl")
eventos_df

In [7]:
from getpass import getpass #Para ingresar la API KEY de OPEN AI
import os #Para cargar la API en las variables de entorno de la máquina
OPENAI_API_KEY = getpass('Enter the secret value: ')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

In [8]:
head_df = eventos_df.head(5).to_string(index=False)

descripcion_df="""
Este DataFrame contiene información acerca de interrupciones o eventos presentadas en redes eléctricas de media tensión,
más específicamente en tres tipos de equipos: Tranformadores, interruptores y tramos de linea (tramos de red).

Las columnas incluyen:
- **Evento**: Id de la interrupción o el evento.
- **equipo_ope**: Código del equipo en el que ocurrió la interrupción.
- **tipo_equi_ope**: Me indica si la interrupción ocurrió sobre un Transformador, o sobre un interruptor o sobre un tramo de linea, es decir que tiene solo tres posibles valores.
- **cto_equi_ope**: Código del circuito al que pertenece el equipo en el cual se dió la interrupción.
- **tipo_elemento**: Capacidad en Kilo Voltios del equipo en el cual ocurrió la interrupción, tiene 4 posibles valores: 33, 13.2, TFD y TFP
- **inicio**: Fecha y hora del inicio del evento o interrupción.
- **fin**: Fecha y hora de la finalización del evento o interrupción.
- **duracion_h**: Duración en horas del evento o interrupción.
- **tipo_duracion**: Variable categórica que indica si ele vento duró más de tres minutos o no; por tanto, tiene dos posibles valores: > 3 min y <= 3 min
- **causa**: Causa del evento o interrupción.
- **CNT_TRAFOS_AFEC**: Cantidad de transformadores afectados en la interrupción o evento.
- **cnt_usus**: Cantidad de usuarios afectados por la interrupción o evento.
- **SAIDI**: Indicador que mide el promedio de la duración en horas de la interrupción por usuario.
- **SAIFI**: Indicador que mide el promedio de cantidad de interrupciones por usuario.
- **PHASES**: Número de fases del equipo en el que ocurrió la interrupción; por tanto tiene 3 posibles valores: 3., 1., 2.
- **FPARENT**: Código del circuito que contiene el equipo en donde se presentó la interrupción.
- **FECHA**: Fecha en la que se presentó el evento o interrupción.
- **LONGITUD**: Longitud geográfica de la ubicación del equipo en el que se presentó la interrupción o evento.
- **LATITUD**: Latiud geográfica de la ubicación del equipo en el que se presentó la interrupción o evento.
- **DEP**: Departamento en donde se presentó la interrupción o evento.
- **MUN**: Municipio en donde se presentó la interrupción o evento.
"""

agent = create_pandas_dataframe_agent(
        ChatOpenAI(temperature=0, model="gpt-3.5-turbo"),
        eventos_df,
        verbose=True,
        agent_type="openai-functions",
        prefix=descripcion_df,  # Añade la descripción al inicio del prompt #suffix=suffix_instrucciones.format(path_plot=path_plot),
        allow_dangerous_code=True)

query="Cuantas interrupciones son superiores a 2 horas por cada año. Respondeme siempre en español."
response=agent.invoke(query)["output"]

/tmp/ipython-input-484/3515689416.py:32: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  ChatOpenAI(temperature=0, model="gpt-3.5-turbo"),




> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['inicio'] = pd.to_datetime(df['inicio'])\ninterrupciones_superiores_2h_por_anio = df[df['duracion_h'] > 2].groupby(df['inicio'].dt.year)['evento'].count()\ninterrupciones_superiores_2h_por_anio"}`


NameError: name 'pd' is not defined
Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\ndf['inicio'] = pd.to_datetime(df['inicio'])\ninterrupciones_superiores_2h_por_anio = df[df['duracion_h'] > 2].groupby(df['inicio'].dt.year)['evento'].count()\ninterrupciones_superiores_2h_por_anio"}`
responded: Se produjo un error al intentar ejecutar el código. Permíteme corregirlo y volver a intentarlo.

inicio
2019    7379
2020    5999
2021    8479
2022    7936
2023    8701
2024    3728
Name: evento, dtype: int64El número de interrupciones superiores a 2 horas por año son:
- 2019: 7379 interrupciones
- 2020: 5999 interrupciones
- 2021: 8479 interrupciones
- 2022: 7936 interrupciones
- 2023: 8701 in

In [ ]:
%%writefile app.py
# pega aquí todo el código anterior
import os
import pandas as pd
import streamlit as st

from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

# ---------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------
st.set_page_config(page_title="Agente con DataFrame", layout="wide")
st.title("Agente de Pandas sobre eventos eléctricos")

# Tu API Key
OPENAI_API_KEY = st.text_input("OpenAI API Key", type="password")

# Ruta del archivo .pkl en Colab
ruta_pkl = st.text_input("Ruta del archivo .pkl", value="eventos_df.pkl")

descripcion_df = """
Este DataFrame contiene información acerca de interrupciones o eventos presentadas en redes eléctricas de media tensión,
más específicamente en tres tipos de equipos: Transformadores, interruptores y tramos de línea.

Las columnas incluyen:
- Evento: Id de la interrupción o el evento.
- equipo_ope: Código del equipo en el que ocurrió la interrupción.
- tipo_equi_ope: Indica si la interrupción ocurrió sobre un Transformador, un interruptor o un tramo de línea.
- cto_equi_ope: Código del circuito al que pertenece el equipo.
- tipo_elemento: Capacidad del equipo. Posibles valores: 33, 13.2, TFD y TFP.
- inicio: Fecha y hora del inicio del evento.
- fin: Fecha y hora del fin del evento.
- duracion_h: Duración en horas.
- tipo_duracion: > 3 min o <= 3 min.
- causa: Causa del evento.
- CNT_TRAFOS_AFEC: Cantidad de transformadores afectados.
- cnt_usus: Cantidad de usuarios afectados.
- SAIDI: Promedio de duración en horas por usuario.
- SAIFI: Promedio de cantidad de interrupciones por usuario.
- PHASES: Número de fases.
- FPARENT: Código del circuito contenedor.
- FECHA: Fecha del evento.
- LONGITUD: Longitud geográfica.
- LATITUD: Latitud geográfica.
- DEP: Departamento.
- MUN: Municipio.
"""

# ---------------------------------------------------
# CARGA DEL DATAFRAME
# ---------------------------------------------------
if OPENAI_API_KEY and ruta_pkl:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

    try:
        eventos_df = pd.read_pickle(ruta_pkl)

        # por si algunas fechas vienen como texto
        for col in ["inicio", "fin", "FECHA"]:
            if col in eventos_df.columns:
                eventos_df[col] = pd.to_datetime(eventos_df[col], errors="coerce")

        st.success("DataFrame cargado correctamente")
        st.write("Dimensión del DataFrame:", eventos_df.shape)
        st.dataframe(eventos_df.head())

        # Crear agente una sola vez
        agent = create_pandas_dataframe_agent(
        ChatOpenAI(temperature=0, model="gpt-3.5-turbo"),
        eventos_df,
        verbose=True,
        agent_type="openai-functions",
        prefix=descripcion_df,  # Añade la descripción al inicio del prompt #suffix=suffix_instrucciones.format(path_plot=path_plot),
        allow_dangerous_code=True)

        # ---------------------------------------------------
        # PESTAÑAS
        # ---------------------------------------------------
        tab1, tab2 = st.tabs(["Preguntas sobre el DataFrame", "Gráficas con agente"])

        # ---------------- TAB 1 ----------------
        with tab1:
            st.subheader("Haz una pregunta al DataFrame")

            pregunta = st.text_area(
                "Escribe tu pregunta",
                value="¿Cuántas interrupciones son superiores a 2 horas por cada año? Responde siempre en español."
            )

            if st.button("Consultar"):
                with st.spinner("Consultando..."):
                    try:
                        respuesta = agent.invoke({"input": pregunta})
                        st.write("### Respuesta")
                        st.write(respuesta["output"])
                    except Exception as e:
                        st.error(f"Error al consultar: {e}")

        # ---------------- TAB 2 ----------------
        with tab2:
            st.subheader("Generar gráfica automáticamente")

            instruccion_grafica = st.text_area(
                "Describe la gráfica",
                value="Construye un gráfico en el cual muestres la cantidad de interrupciones por cada departamento."
            )

            if st.button("Generar gráfica"):
                with st.spinner("Generando gráfica..."):
                    try:
                        path_plot = "grafica_streamlit.png"

                        prompt_grafica = f"""
                        {instruccion_grafica}

                        Construye el gráfico de la forma más estética posible para mostrar a un usuario.
                        Puedes utilizar colores verde y gris en diferentes tonalidades.
                        Los títulos y ejes del gráfico deben estar en español.
                        Guarda la imagen en la ruta relativa {path_plot}.
                        No ejecutes plt.show().
                        Siempre ejecuta plt.tight_layout().
                        Responde siempre en español.
                        """

                        respuesta = agent.invoke({"input": prompt_grafica})

                        st.write("### Mensaje del agente")
                        st.write(respuesta["output"])

                        if os.path.exists(path_plot):
                            st.image(path_plot, caption="Gráfica generada")
                        else:
                            st.warning("El agente respondió, pero no encontré la imagen guardada.")

                    except Exception as e:
                        st.error(f"Error al generar la gráfica: {e}")

    except Exception as e:
        st.error(f"No se pudo leer el archivo .pkl: {e}")
else:
    st.info("Ingresa tu API Key y la ruta del archivo .pkl")

In [ ]:
if curso_setup.en_colab():
    !wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared-linux-amd64
    !mv cloudflared-linux-amd64 /usr/local/bin/cloudflared



In [ ]:
def run():
    if not curso_setup.en_colab():
        print("[curso] Esta celda solo funciona en Google Colab.")
        return
    # Ejecutar Streamlit
    !streamlit run app.py &>logs.txt &

    # Exponer el puerto 8501 con Cloudflare Tunnel
    !cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1 &

    # Leer la URL pública generada por Cloudflare usando '|' como guía (con fallback)
    import time, re

    time.sleep(5)  # breve espera

    url = None
    with open('cloudflared.log', encoding='utf-8', errors='ignore') as f:
        for line in f:
            # 1) Intento principal: tomar el texto entre barras verticales
            if 'trycloudflare.com' in line and '|' in line:
                # Divide por '|' y busca el fragmento que empiece con http
                candidates = [p.strip() for p in line.split('|') if p.strip()]
                for c in candidates:
                    if c.startswith('http') and 'trycloudflare.com' in c:
                        url = c
                        break
                if url:
                    break

            # 2) Respaldo: regex por si el formato cambia
            if 'trycloudflare.com' in line and url is None:
                m = re.search(r"https?://[^\s]*trycloudflare\.com[^\s]*", line)
                if m:
                    url = m.group(0).strip(' |')
                    break

    if url:
        print(f"Tu aplicación está disponible en: {url}")
    else:
        print("No encontré la URL aún. Prueba aumentar el time.sleep o revisa el log con:")
        print("!tail -n 40 cloudflared.log")

In [15]:
run()

Tu aplicación está disponible en: https://adrian-suppliers-anyway-indexes.trycloudflare.com


In [ ]:
if curso_setup.en_colab():
    !pkill streamlit